In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 McNemar's Test: Best-Baseline vs Parameter-Free vs P-Dependent Fusion
 Pooled 5-fold TEST set, seed=62.
================================================================================

For each of the 8 fusion systems, this compares THREE classifiers on the
SAME held-out test trials, using McNemar's test (the correct statistical test
for two classifiers evaluated on paired/matched trials, as opposed to an
unpaired test that would ignore the fact that all three systems are scored
on the identical subjects):

  1. Best-Baseline : whichever single constituent modality of that system
                      (e.g. Face_C or Face_G for S1) has the lower standalone
                      EER on pooled TEST data. The natural "is fusion even
                      worth it" reference point.
  2. Param-Free     : that system's winning parameter-free rule (no p),
                      selected earlier from TRAIN.
  3. P-Dependent    : that system's winning (rule, p), selected earlier
                      from TRAIN.

Two kinds of paired binary decision are tested per system, per pair of
classifiers:

  EER-decision      : at each classifier's own EER threshold (computed from
                       its pooled TEST scores), was a genuine trial accepted
                       / an impostor trial rejected? ("correct" or not).
  Rank-1@FAR=1%      : for each probe, did the classifier's top match name
  decision            the right identity AND clear the FAR=1% threshold?

McNemar's test only uses the trials where the two classifiers DISAGREE (one
correct, one wrong) -- it asks whether disagreements favour one classifier
significantly more than chance, which is exactly the right question when
deciding whether fusion (of either family) is a real improvement over the
best single modality, or over the other fusion family.

OUTPUT
------
McNemar_Comparison_seed62.xlsx with:
  Methodology
  Summary            - one row per (System, Metric, Comparison) with the
                       McNemar statistic, p-value, and which side won
  Raw_Contingency     - the full 2x2 discordant-pair counts behind every test
================================================================================
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy import stats
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.worksheet import Worksheet


# ==============================================================================
#  CONFIGURATION
# ==============================================================================

MAT_FILE = "870d37a7-673d-49ff-8a98-47b9b519237f.mat"
SEED = 62
N = 517
N_FOLDS = 5
FAR_TARGET = 0.01
OUTPUT_PATH = f"./Results/McNemar_Comparison_seed{SEED}.xlsx"
THEME_COLOR = "8E24AA"
FONT_NAME = "Arial"

RAW_KEYS = {
    "Face_C":    "fingToFaceCSCORES",
    "Face_G":    "fingToFaceGSCORES",
    "Finger_LI": "VfingToFaceliSCORES",
    "Finger_RI": "VfingToFaceriSCORES",
}

SYSTEM_DEFS = {
    "S1": dict(stage1=("Face_C", "Face_G"), cascaded=False),
    "S2": dict(stage1=("Face_C", "Finger_LI"), cascaded=False),
    "S3": dict(stage1=("Face_C", "Finger_RI"), cascaded=False),
    "S4": dict(stage1=("Face_G", "Finger_LI"), cascaded=False),
    "S5": dict(stage1=("Face_G", "Finger_RI"), cascaded=False),
    "S6": dict(stage1=("Finger_LI", "Finger_RI"), cascaded=False),
    "S7": dict(stage1=("Finger_LI", "Finger_RI"), stage2="Face_C", cascaded=True),
    "S8": dict(stage1=("Finger_LI", "Finger_RI"), stage2="Face_G", cascaded=True),
}
SYSTEMS = list(SYSTEM_DEFS.keys())

# Winning configurations, selected earlier from TRAIN (seed=62).
PF_WINNERS: Dict[str, str] = {
    "S1": "Rule3", "S2": "Rule5", "S3": "Rule5", "S4": "Rule5",
    "S5": "Rule5", "S6": "Rule8", "S7": "Rule5", "S8": "Rule5",
}
PDEP_WINNERS: Dict[str, Tuple[str, float]] = {
    "S1": ("R5p", 0.40), "S2": ("Werners1", 0.45), "S3": ("R5p", 0.25),
    "S4": ("R5p", 0.30), "S5": ("Luhandjula", 0.45), "S6": ("Werners2", 0.45),
    "S7": ("Mizumoto2", 0.75), "S8": ("Luhandjula", 0.80),
}


# ==============================================================================
#  FUSION RULE FORMULAS
# ==============================================================================

def apply_pdep_rule(rule: str, x: np.ndarray, y: np.ndarray, p: float) -> np.ndarray:
    valid = (x != -1) & (y != -1)
    xv = np.where(valid, x, 0.0)
    yv = np.where(valid, y, 0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        if rule == "Zimmermann1":
            xy = xv * yv
            s = xv + yv - xy
            xy_c, s_c = np.clip(xy, 0, None), np.clip(s, 0, None)
            t1 = np.power(xy_c, 1 - p, where=(xy_c > 0), out=np.zeros_like(xy_c))
            t1 = np.where(xy_c == 0, 0.0 ** (1 - p) if (1 - p) != 0 else 1.0, t1)
            t2 = np.power(s_c, p, where=(s_c > 0), out=np.zeros_like(s_c))
            t2 = np.where(s_c == 0, 0.0 ** p if p != 0 else 1.0, t2)
            F = t1 * t2
        elif rule == "Zimmermann2": F = np.minimum(xv, yv) * (1 - p) + np.maximum(xv, yv) * p
        elif rule == "Luhandjula":  F = np.minimum(xv, yv) * (1 - p) + np.minimum(1, xv + yv) * p
        elif rule == "Werners1":    F = np.minimum(xv, yv) * (1 - p) + ((xv + yv) / 2) * p
        elif rule == "Werners2":    F = np.maximum(xv, yv) * (1 - p) + ((xv + yv) / 2) * p
        elif rule == "Sales1":      F = np.maximum(0, xv + yv - 1) * (1 - p) + np.minimum(xv, yv) * p
        elif rule == "Sales2":      F = np.minimum(1, xv + yv) * (1 - p) + np.maximum(xv, yv) * p
        elif rule == "Mizumoto1":   F = np.minimum(xv, yv) * (1 - p) + np.sqrt(np.clip(xv * yv, 0, None)) * p
        elif rule == "Mizumoto2":   F = ((xv + yv) / 2) * (1 - p) + np.sqrt(np.clip(xv * yv, 0, None)) * p
        elif rule == "R5p":
            s = xv + yv - xv * yv
            F = p * s + (1 - p) * (1 - s) * np.minimum(xv, yv)
        else:
            raise ValueError(rule)
    F = np.where(valid, F, -1.0)
    return np.clip(F, np.where(valid, 0, -1), 1)


def apply_pf_rule(rule: str, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    valid = (x != -1) & (y != -1)
    xv = np.where(valid, x, 0.0)
    yv = np.where(valid, y, 0.0)
    xy = xv * yv
    mn, mx = np.minimum(xv, yv), np.maximum(xv, yv)
    s = xv + yv - xy
    if rule == "Rule1": F = xy * mx
    elif rule == "Rule2": F = xy * s
    elif rule == "Rule3": F = xy * mn
    elif rule == "Rule4": F = xy + xy ** 2 - xy * mn
    elif rule == "Rule5": F = s + mn - s * mn
    elif rule == "Rule6": F = (xv + yv) - xy * s
    elif rule == "Rule7": F = s + mx - s * mx
    elif rule == "Rule8": F = s * mx
    else: raise ValueError(rule)
    return np.where(valid, np.clip(F, 0, 1), -1.0)


def build_pdep_matrix(system: str, mods: Dict[str, np.ndarray]) -> np.ndarray:
    sdef = SYSTEM_DEFS[system]
    rule, p = PDEP_WINNERS[system]
    a, b = sdef["stage1"]
    fused = apply_pdep_rule(rule, mods[a], mods[b], p)
    if sdef["cascaded"]:
        fused = apply_pdep_rule(rule, fused, mods[sdef["stage2"]], p)
    return fused


def build_pf_matrix(system: str, mods: Dict[str, np.ndarray]) -> np.ndarray:
    sdef = SYSTEM_DEFS[system]
    rule = PF_WINNERS[system]
    a, b = sdef["stage1"]
    fused = apply_pf_rule(rule, mods[a], mods[b])
    if sdef["cascaded"]:
        fused = apply_pf_rule(rule, fused, mods[sdef["stage2"]])
    return fused


def constituent_modalities(system: str) -> List[str]:
    sdef = SYSTEM_DEFS[system]
    mods = list(sdef["stage1"])
    if sdef["cascaded"]:
        mods.append(sdef["stage2"])
    return mods


# ==============================================================================
#  NORMALIZATION
# ==============================================================================

def minmax_fit_transform(S_full: np.ndarray, train_ids: np.ndarray,
                         test_ids: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    S_full = S_full.astype(float)
    S_train = S_full[np.ix_(train_ids, train_ids)]
    S_test = S_full[np.ix_(test_ids, test_ids)]
    invalid = (S_train == -1)
    valid_vals = S_train[~invalid]
    vmin, vmax = valid_vals.min(), valid_vals.max()

    def transform(S):
        inv = (S == -1)
        out = np.full_like(S, -1.0)
        out[~inv] = (S[~inv] - vmin) / (vmax - vmin)
        return out

    return transform(S_train), transform(S_test)


# ==============================================================================
#  BINARY DECISION EXTRACTION
# ==============================================================================

def eer_threshold(genuine: np.ndarray, impostor: np.ndarray) -> float:
    G, I = np.sort(genuine), np.sort(impostor)
    thr = np.unique(np.concatenate([G, I]))
    far = 1 - np.searchsorted(I, thr, side="left") / len(I)
    frr = np.searchsorted(G, thr, side="left") / len(G)
    idx = np.argmin(np.abs(far - frr))
    return float(thr[idx])


def eer_decisions(fold_matrices: List[np.ndarray]) -> Tuple[Dict[Tuple[int, int, int], bool], float]:
    """
    For every (fold, row, col) valid trial across the 5 folds, return whether
    the classifier decided CORRECTLY at its own (pooled) EER threshold:
    genuine trial correct if score >= threshold; impostor trial correct if
    score < threshold. Keyed by (fold_index, row, col) so two classifiers'
    decisions can be matched trial-for-trial.
    """
    all_genuine, all_impostor = [], []
    for S in fold_matrices:
        diag = np.diag(S)
        off = ~np.eye(S.shape[0], dtype=bool)
        all_genuine.append(diag[diag != -1])
        imp = S[off]
        all_impostor.append(imp[imp != -1])
    genuine_pool = np.concatenate(all_genuine)
    impostor_pool = np.concatenate(all_impostor)
    threshold = eer_threshold(genuine_pool, impostor_pool)

    decisions: Dict[Tuple[int, int, int], bool] = {}
    for f, S in enumerate(fold_matrices):
        n = S.shape[0]
        for i in range(n):
            for j in range(n):
                v = S[i, j]
                if v == -1:
                    continue
                is_genuine = i == j
                correct = (v >= threshold) if is_genuine else (v < threshold)
                decisions[(f, i, j)] = bool(correct)
    return decisions, threshold


def rank1_far_decisions(fold_matrices: List[np.ndarray],
                        far_target: float = FAR_TARGET) -> Tuple[Dict[Tuple[int, int], bool], float]:
    """
    For every (fold, probe_row) with a valid genuine mate, return whether the
    classifier's top match was correct AND cleared the FAR=1% threshold
    (computed from that classifier's own pooled impostor distribution).
    """
    all_impostor = []
    for S in fold_matrices:
        off = ~np.eye(S.shape[0], dtype=bool)
        imp = S[off]
        all_impostor.append(imp[imp != -1])
    impostor_pool = np.concatenate(all_impostor)
    threshold = np.percentile(impostor_pool, 100 * (1 - far_target))

    decisions: Dict[Tuple[int, int], bool] = {}
    for f, S in enumerate(fold_matrices):
        n = S.shape[0]
        diag = np.diag(S)
        M = np.where(S == -1, -np.inf, S)
        top_score = M.max(axis=1)
        top_idx = M.argmax(axis=1)
        for i in range(n):
            if diag[i] == -1:
                continue
            correct = bool((top_idx[i] == i) and (top_score[i] >= threshold))
            decisions[(f, i)] = correct
    return decisions, threshold


# ==============================================================================
#  MCNEMAR'S TEST
# ==============================================================================

def mcnemar_test(decisions_a: Dict, decisions_b: Dict) -> dict:
    """
    Build the 2x2 discordant-pair contingency table from two matched binary
    decision dictionaries (same keys = same trials) and compute McNemar's
    test with continuity correction.

    b = # trials A correct, B wrong
    c = # trials A wrong,  B correct
    statistic = (|b-c| - 1)^2 / (b+c)   [0 if b+c == 0]
    p-value from chi2 with 1 degree of freedom
    """
    common_keys = set(decisions_a) & set(decisions_b)
    both_correct = b_count = c_count = neither = 0
    for key in common_keys:
        da, db = decisions_a[key], decisions_b[key]
        if da and db:
            both_correct += 1
        elif da and not db:
            b_count += 1   # A correct, B wrong
        elif not da and db:
            c_count += 1   # A wrong, B correct
        else:
            neither += 1

    n_common = len(common_keys)
    if (b_count + c_count) == 0:
        return dict(n_trials=n_common, both_correct=both_correct, neither_correct=neither,
                   A_correct_B_wrong=b_count, A_wrong_B_correct=c_count,
                   statistic=0.0, p_value=1.0, winner="tie (no discordant trials)")

    statistic = (abs(b_count - c_count) - 1) ** 2 / (b_count + c_count)
    p_value = float(1 - stats.chi2.cdf(statistic, df=1))
    if p_value < 0.05:
        winner = "B" if c_count > b_count else "A"
    else:
        winner = "no significant difference"

    return dict(n_trials=n_common, both_correct=both_correct, neither_correct=neither,
               A_correct_B_wrong=b_count, A_wrong_B_correct=c_count,
               statistic=round(statistic, 4), p_value=round(p_value, 6), winner=winner)


# ==============================================================================
#  MAIN STUDY
# ==============================================================================

def run_comparison() -> Tuple[pd.DataFrame, pd.DataFrame]:
    contents = loadmat(MAT_FILE)
    raw_matrices = {mod: np.asarray(contents[key], dtype=float) for mod, key in RAW_KEYS.items()}

    rng = np.random.default_rng(SEED)
    perm = rng.permutation(N)
    folds = np.array_split(perm, N_FOLDS)

    # Per-fold clean TEST submatrices for all 4 base modalities.
    clean_test_per_fold: List[Dict[str, np.ndarray]] = []
    for fold_idx in range(N_FOLDS):
        test_ids = np.sort(folds[fold_idx])
        train_ids = np.sort(np.setdiff1d(np.arange(N), test_ids))
        fold_mods = {}
        for mod, matrix in raw_matrices.items():
            _, test_block = minmax_fit_transform(matrix, train_ids, test_ids)
            fold_mods[mod] = test_block
        clean_test_per_fold.append(fold_mods)

    summary_rows = []
    contingency_rows = []

    for system in SYSTEMS:
        # ---- Build the 3 classifiers' per-fold matrices ----
        pdep_mats = [build_pdep_matrix(system, clean_test_per_fold[f]) for f in range(N_FOLDS)]
        pf_mats = [build_pf_matrix(system, clean_test_per_fold[f]) for f in range(N_FOLDS)]

        # Best baseline: whichever constituent modality has lower standalone EER.
        candidates = constituent_modalities(system)
        best_mod, best_mod_eer = None, np.inf
        for mod in candidates:
            mats = [clean_test_per_fold[f][mod] for f in range(N_FOLDS)]
            genuine_all, impostor_all = [], []
            for S in mats:
                diag = np.diag(S)
                off = ~np.eye(S.shape[0], dtype=bool)
                genuine_all.append(diag[diag != -1])
                imp = S[off]
                impostor_all.append(imp[imp != -1])
            g, i = np.concatenate(genuine_all), np.concatenate(impostor_all)
            thr = eer_threshold(g, i)
            far = (i >= thr).mean()
            frr = (g < thr).mean()
            eer_val = (far + frr) / 2
            if eer_val < best_mod_eer:
                best_mod_eer = eer_val
                best_mod = mod
        baseline_mats = [clean_test_per_fold[f][best_mod] for f in range(N_FOLDS)]

        classifiers = {"Baseline": baseline_mats, "ParamFree": pf_mats, "PDep": pdep_mats}
        labels = {"Baseline": f"Best-Baseline ({best_mod})",
                 "ParamFree": f"Param-Free ({PF_WINNERS[system]})",
                 "PDep": f"Param-Dep ({PDEP_WINNERS[system][0]}, p={PDEP_WINNERS[system][1]})"}

        # ---- EER-decision McNemar tests ----
        eer_decisions_by_clf = {}
        for name, mats in classifiers.items():
            dec, thr = eer_decisions(mats)
            eer_decisions_by_clf[name] = dec

        # ---- Rank-1@FAR=1%-decision McNemar tests ----
        r1_decisions_by_clf = {}
        for name, mats in classifiers.items():
            dec, thr = rank1_far_decisions(mats)
            r1_decisions_by_clf[name] = dec

        pairs = [("Baseline", "ParamFree"), ("Baseline", "PDep"), ("ParamFree", "PDep")]
        for a_name, b_name in pairs:
            # EER-based
            result = mcnemar_test(eer_decisions_by_clf[a_name], eer_decisions_by_clf[b_name])
            acc_a = np.mean(list(eer_decisions_by_clf[a_name].values())) * 100
            acc_b = np.mean(list(eer_decisions_by_clf[b_name].values())) * 100
            winner_label = {"A": labels[a_name], "B": labels[b_name],
                           "tie (no discordant trials)": "tie",
                           "no significant difference": "no significant difference"}[result["winner"]]
            summary_rows.append(dict(
                System=system, Metric="EER-decision",
                A=labels[a_name], B=labels[b_name],
                Accuracy_A=round(acc_a, 4), Accuracy_B=round(acc_b, 4),
                N_trials=result["n_trials"],
                Discordant_A_correct=result["A_correct_B_wrong"],
                Discordant_B_correct=result["A_wrong_B_correct"],
                McNemar_stat=result["statistic"], p_value=result["p_value"],
                Significant_0_05=result["p_value"] < 0.05, Winner=winner_label,
            ))
            contingency_rows.append(dict(
                System=system, Metric="EER-decision", A=labels[a_name], B=labels[b_name],
                Both_correct=result["both_correct"], Neither_correct=result["neither_correct"],
                A_correct_B_wrong=result["A_correct_B_wrong"],
                A_wrong_B_correct=result["A_wrong_B_correct"],
                N_trials=result["n_trials"],
            ))

            # Rank-1@FAR=1%-based
            result = mcnemar_test(r1_decisions_by_clf[a_name], r1_decisions_by_clf[b_name])
            acc_a = np.mean(list(r1_decisions_by_clf[a_name].values())) * 100
            acc_b = np.mean(list(r1_decisions_by_clf[b_name].values())) * 100
            winner_label = {"A": labels[a_name], "B": labels[b_name],
                           "tie (no discordant trials)": "tie",
                           "no significant difference": "no significant difference"}[result["winner"]]
            summary_rows.append(dict(
                System=system, Metric="Rank1@FAR1%-decision",
                A=labels[a_name], B=labels[b_name],
                Accuracy_A=round(acc_a, 4), Accuracy_B=round(acc_b, 4),
                N_trials=result["n_trials"],
                Discordant_A_correct=result["A_correct_B_wrong"],
                Discordant_B_correct=result["A_wrong_B_correct"],
                McNemar_stat=result["statistic"], p_value=result["p_value"],
                Significant_0_05=result["p_value"] < 0.05, Winner=winner_label,
            ))
            contingency_rows.append(dict(
                System=system, Metric="Rank1@FAR1%-decision", A=labels[a_name], B=labels[b_name],
                Both_correct=result["both_correct"], Neither_correct=result["neither_correct"],
                A_correct_B_wrong=result["A_correct_B_wrong"],
                A_wrong_B_correct=result["A_wrong_B_correct"],
                N_trials=result["n_trials"],
            ))

        print(f"  {system} done. Best baseline modality: {best_mod} (EER={best_mod_eer*100:.2f}%)")

    return pd.DataFrame(summary_rows), pd.DataFrame(contingency_rows)


# ==============================================================================
#  EXCEL OUTPUT
# ==============================================================================

def _styles():
    thin = Side(style="thin", color="D9D9D9")
    return {
        "header_fill": PatternFill("solid", fgColor=THEME_COLOR),
        "header_font": Font(name=FONT_NAME, bold=True, color="FFFFFF", size=10),
        "title_font":  Font(name=FONT_NAME, bold=True, size=13, color=THEME_COLOR),
        "note_font":   Font(name=FONT_NAME, italic=True, size=9, color="555555"),
        "cell_font":   Font(name=FONT_NAME, size=10),
        "border":      Border(left=thin, right=thin, top=thin, bottom=thin),
    }


def _autofit(worksheet: Worksheet, n_columns: int, header_row: int, n_rows: int,
            minimum: int = 9, maximum: int = 34) -> None:
    for col_idx in range(1, n_columns + 1):
        letter = get_column_letter(col_idx)
        longest = max(
            (len(str(worksheet.cell(row=r, column=col_idx).value))
             for r in range(header_row, header_row + n_rows + 1)
             if worksheet.cell(row=r, column=col_idx).value is not None),
            default=minimum,
        )
        worksheet.column_dimensions[letter].width = min(max(longest + 2, minimum), maximum)


def _write_table(worksheet: Worksheet, frame: pd.DataFrame, styles: dict,
                 title: str, subtitle: str = "") -> int:
    worksheet.cell(row=1, column=1, value=title).font = styles["title_font"]
    row = 2
    if subtitle:
        worksheet.cell(row=row, column=1, value=subtitle).font = styles["note_font"]
        row += 1
    row += 1
    header_row = row
    for col_idx, column in enumerate(frame.columns, start=1):
        cell = worksheet.cell(row=header_row, column=col_idx, value=str(column))
        cell.fill = styles["header_fill"]
        cell.font = styles["header_font"]
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = styles["border"]
    for offset, record in enumerate(frame.itertuples(index=False), start=1):
        for col_idx, value in enumerate(record, start=1):
            if isinstance(value, (np.integer,)):
                value = int(value)
            elif isinstance(value, (np.floating,)):
                value = None if np.isnan(value) else float(value)
            elif isinstance(value, (np.bool_,)):
                value = bool(value)
            cell = worksheet.cell(row=header_row + offset, column=col_idx, value=value)
            cell.font = styles["cell_font"]
            cell.border = styles["border"]
    worksheet.freeze_panes = worksheet.cell(row=header_row + 1, column=1)
    _autofit(worksheet, len(frame.columns), header_row, len(frame))
    return header_row


def _write_methodology(worksheet: Worksheet, styles: dict) -> None:
    worksheet.column_dimensions["A"].width = 112
    lines = [
        ("title", "Methodology"),
        ("note",  "McNemar's Test: Best-Baseline vs Parameter-Free vs P-Dependent Fusion"),
        ("blank", ""),
        ("head",  "WHY McNEMAR'S TEST"),
        ("body",  "All three classifiers (best baseline, parameter-free fusion, p-dependent fusion) "
                  "are evaluated on the SAME trials -- the same genuine/impostor comparisons, pooled "
                  "across the 5 TEST folds (seed=62). This is a paired/matched-samples setting, so an "
                  "unpaired test (e.g. a simple difference-of-proportions test) would be statistically "
                  "wrong: it would ignore that both classifiers succeed or fail on the SAME underlying "
                  "trial, not on independent samples. McNemar's test is designed exactly for this case."),
        ("body",  "McNemar's test only looks at DISCORDANT trials -- where one classifier is correct "
                  "and the other is wrong -- and asks whether that disagreement favours one classifier "
                  "significantly more often than chance (i.e. more than a 50/50 split of the "
                  "disagreements). Trials where both classifiers agree (both correct or both wrong) "
                  "carry no information about which is better and are excluded from the statistic."),
        ("blank", ""),
        ("head",  "THE THREE CLASSIFIERS"),
        ("body",  "Best-Baseline: for each system, whichever single constituent modality (e.g. Face_C "
                  "or Face_G for S1) has the LOWER standalone EER on pooled TEST data. This is the "
                  "natural 'is fusion worth it at all' reference point."),
        ("body",  "Param-Free: that system's winning parameter-free rule (selected earlier from TRAIN, "
                  "no p involved)."),
        ("body",  "P-Dependent: that system's winning (rule, p) pair (selected earlier from TRAIN)."),
        ("blank", ""),
        ("head",  "TWO KINDS OF BINARY DECISION TESTED"),
        ("body",  "EER-decision: each classifier's own pooled-TEST EER threshold is computed. A "
                  "genuine trial is 'correct' if its score clears that threshold (accepted); an "
                  "impostor trial is 'correct' if its score falls below it (rejected). This tests "
                  "verification performance."),
        ("body",  "Rank-1@FAR=1%-decision: each classifier's own FAR=1% threshold is computed from its "
                  "pooled impostor scores. A probe is 'correct' if its top match is the right identity "
                  "AND clears that threshold. This tests identification performance."),
        ("blank", ""),
        ("head",  "READING THE SUMMARY SHEET"),
        ("body",  "McNemar_stat and p_value: p_value < 0.05 means the difference between classifiers A "
                  "and B is statistically significant at the 5% level (not just a random fluctuation in "
                  "which trials happened to disagree). Winner names whichever classifier won the "
                  "significant majority of discordant trials, or reports 'no significant difference' "
                  "if p >= 0.05 -- in that case, any accuracy gap between A and B should NOT be treated "
                  "as a reliable improvement, since it is within the range explainable by chance "
                  "disagreement on which trials went which way."),
        ("body",  "Discordant_A_correct / Discordant_B_correct: the raw counts behind the test -- how "
                  "many trials A got right and B got wrong, and vice versa. These two numbers are what "
                  "McNemar's statistic is actually computed from."),
    ]
    fonts = {
        "title": styles["title_font"], "head": Font(name=FONT_NAME, bold=True, size=11),
        "note": styles["note_font"], "body": styles["cell_font"], "blank": styles["cell_font"],
    }
    for row, (kind, text) in enumerate(lines, start=1):
        cell = worksheet.cell(row=row, column=1, value=text)
        cell.font = fonts[kind]
        cell.alignment = Alignment(wrap_text=False, vertical="top")


def build_workbook(summary: pd.DataFrame, contingency: pd.DataFrame, output_path: Path) -> Path:
    workbook = Workbook()
    styles = _styles()

    ws = workbook.active
    ws.title = "Methodology"
    _write_methodology(ws, styles)

    ws = workbook.create_sheet("Summary")
    _write_table(ws, summary, styles,
                title="McNemar's Test Summary \u2014 Best-Baseline vs Param-Free vs P-Dependent",
                subtitle="48 tests: 8 systems x 3 classifier pairs x 2 metrics (EER-decision, Rank-1@FAR=1%-decision).")

    ws = workbook.create_sheet("Raw_Contingency")
    _write_table(ws, contingency, styles,
                title="Raw 2x2 Contingency Counts Behind Every McNemar Test",
                subtitle="Both_correct + Neither_correct + A_correct_B_wrong + A_wrong_B_correct = N_trials.")

    workbook.save(output_path)
    return Path(output_path)


# ==============================================================================
#  ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    print(f"Running McNemar comparison: seed={SEED}, {N_FOLDS} folds ...")
    summary, contingency = run_comparison()
    print(f"\n{len(summary)} McNemar tests computed.")

    output_path = build_workbook(summary, contingency, Path(OUTPUT_PATH))
    print(f"Workbook written to: {output_path}")

    print("\n=== Significant differences (p < 0.05) ===")
    sig = summary[summary.Significant_0_05]
    print(sig[["System", "Metric", "A", "B", "p_value", "Winner"]].to_string(index=False))




Running McNemar comparison: seed=42, 5 folds ...
  S1 done. Best baseline modality: Face_C (EER=4.45%)
  S2 done. Best baseline modality: Face_C (EER=4.45%)
  S3 done. Best baseline modality: Face_C (EER=4.45%)
  S4 done. Best baseline modality: Face_G (EER=5.80%)
  S5 done. Best baseline modality: Finger_RI (EER=4.85%)
  S6 done. Best baseline modality: Finger_RI (EER=4.85%)
  S7 done. Best baseline modality: Face_C (EER=4.45%)
  S8 done. Best baseline modality: Finger_RI (EER=4.85%)

48 McNemar tests computed.
Workbook written to: Results\McNemar_Comparison_seed_42.xlsx

=== Significant differences (p < 0.05) ===
System               Metric                         A                              B  p_value                         Winner
    S1         EER-decision    Best-Baseline (Face_C)             Param-Free (Rule3) 0.000000             Param-Free (Rule3)
    S1         EER-decision    Best-Baseline (Face_C)         Param-Dep (R5p, p=0.4) 0.000000         Param-Dep (R5p, p=0.4)
  